<a href="https://colab.research.google.com/github/Davikky/rptu-courses/blob/main/JC_model_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Scientic Computing (JC - Model) -v2

The Hamiltonian describing this system is:

$$
H = \frac{1}{2}\omega_{q}\sigma_{z} + \omega_{0}a^{\dagger}a
    + g(\sigma_{-}a^{\dagger}\, +\, \sigma_{+}a).
$$

In [4]:
!pip install qutip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 55.8 MB/s eta 0:00:00


## Part I: Adiabatic tracking

Here, eigenvalues are sorted by energy at each $ω_0$:

At avoided crossing, **index ordering breaks** hence the need to track eigenstates by **maximum overlap**, not by index.

Given eigenstates:

$$
\{|\psi_{i}(\omega_k)⟩\}, \{|\psi_{j}(\omega_{k+1})⟩\}
$$

State $i$ is tracked by maximizing:
$$
\arg \max_j |⟨\psi_{i}(\omega_k)⟩ |\psi_{j}(\omega_{k+1})⟩ |
$$

This is the adiabatic continuation principle.


### 2.1 Eigenstate tracking

In [5]:
# Eigenstate tracking function
def track_eigenstates(N, g, omega_list, ref_index=0):
    """
    Track an eigenstate across omega_list using maximal overlap.
    """
    energies_all = []
    states_tracked = []

    # initial diagonalization
    energies, states = jc_model(N, g, omega_list[0])
    psi_prev = states[ref_index]

    energies_all.append(energies[ref_index])
    states_tracked.append(psi_prev)

    for omega in omega_list[1:]:
        energies, states = jc_model(N, g, omega)

        overlaps = np.array([
            abs(psi_prev.dag() * psi) for psi in states
        ])

        j = np.argmax(overlaps)
        psi_prev = states[j]

        energies_all.append(energies[j])
        states_tracked.append(psi_prev)

    return np.array(energies_all), states_tracked


### 2.2 Occupation extraction

In [6]:
# Occupation extraction
def tracked_occupation(N, g, omega_list, ref_index):
    dim = 2 * N
    occupation = [[] for _ in range(dim)]

    _, states_tracked = track_eigenstates(N, g, omega_list, ref_index)

    for psi in states_tracked:
        coeffs = psi.full().flatten()
        for i in range(dim):
            occupation[i].append(coeffs[i])

    return occupation


## Part II: Time-dependent Schrödinger equation (TDSE)

Here we switch from eigen-analysis to **dynamics**.


### 2.3 Time-dependent Hamiltonian

Example: chirped cavity frequency

$$
\omega_0(t) = \omega_i + \alpha t
$$

### Hamiltonian decomposition

In [7]:
# Hamiltonian decomposition

def jc_operators(N):
    a  = tensor(identity(2), destroy(N))
    sm = tensor(0.5*(sigmax() - 1j*sigmay()), identity(N))
    sz = tensor(sigmaz(), identity(N))

    return a, sm, sz


In [8]:
# static-Hamitonian definition
def H_static(N, g):
    a, sm, sz = jc_operators(N)
    return 0.5 * sz + g * (sm * a.dag() + sm.dag() * a)


In [9]:
def H_omega(N):
    a, _, _ = jc_operators(N)
    return a.dag() * a


### 2.4 Time-dependent Hamiltonian for `mesolve`

In [10]:
def omega_t(t, args):
    return args['omega_i'] + args['alpha'] * t


In [11]:
def jc_td_hamiltonian(N, g):
    return [
        H_static(N, g),
        [H_omega(N), omega_t]
    ]


### 2.5 Initial state (adiabatic eigenstate)

We shall use the tracked eigenstate at *t = 0*:

In [12]:
omega_list = [omega_i]
_, states = jc_model(N, g, omega_i)
psi0 = states[ref_index]


NameError: name 'omega_i' is not defined

In [ ]:
# solve TDSE
from qutip import mesolve

args = {
    'omega_i': 0.5,
    'alpha': 0.02
}

tlist = np.linspace(0, 100, 400)

H_td = jc_td_hamiltonian(N, g)

result = mesolve(
    H_td,
    psi0,
    tlist,
    c_ops=[],
    e_ops=[]
)


In [ ]:
# Time-dependent occupation
def time_occupation(result, N):
    dim = 2 * N
    occupation_t = [[] for _ in range(dim)]

    for psi in result.states:
        coeffs = psi.full().flatten()
        for i in range(dim):
            occupation_t[i].append(abs(coeffs[i])**2)

    return occupation_t


## v2 - Full Widget Implementation


In [13]:
# imports
from qutip import *
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


In [14]:
# Basis labels (same as v1)
def basis_labels(N):
    return [f"e{n}" for n in range(N)] + [f"g{n}" for n in range(N)]


In [15]:
# Jaynes–Cummings Hamiltonian (static)
def jc_model(N, g, omega_0):
    a  = tensor(identity(2), destroy(N))
    sm = tensor(0.5*(sigmax() - 1j*sigmay()), identity(N))
    sz = tensor(sigmaz(), identity(N))

    H = 0.5 * sz + omega_0 * a.dag() * a + g * (sm * a.dag() + sm.dag() * a)
    return H.eigenstates()


In [20]:
# Eigenstate tracking across avoided crossings

def track_eigenstates(N, g, omega_list, ref_index):
    energies_tracked = []
    states_tracked = []

    energies, states = jc_model(N, g, omega_list[0])
    psi_prev = states[ref_index]

    energies_tracked.append(energies[ref_index])
    states_tracked.append(psi_prev)

    for omega in omega_list[1:]:
        energies, states = jc_model(N, g, omega)

        overlaps = np.array([
            abs(psi_prev.dag() * psi) for psi in states
        ])

        j = np.argmax(overlaps)
        psi_prev = states[j]

        energies_tracked.append(energies[j])
        states_tracked.append(psi_prev)

    return np.array(energies_tracked), states_tracked


In [17]:
# Occupation from tracked eigenstate
def tracked_occupation(N, g, omega_list, ref_index):
    dim = 2 * N
    occupation = [[] for _ in range(dim)]

    _, states_tracked = track_eigenstates(N, g, omega_list, ref_index)

    for psi in states_tracked:
        coeffs = psi.full().flatten()
        for i in range(dim):
            occupation[i].append(coeffs[i])

    return occupation


In [21]:
# Static tracked-state plot (with widgets)
def plot_tracked_components(N, g, ref_index, selected_components, omega_list):
    labels = basis_labels(N)
    occupation = tracked_occupation(N, g, omega_list, ref_index)

    plt.figure(figsize=(7,5))
    for i in selected_components:
        plt.plot(
            omega_list,
            np.abs(occupation[i])**2,
            label=labels[i]
        )

    plt.xlabel(r'$\omega_0$')
    plt.ylabel(r'$|c_i|^2$')
    plt.title(f'Tracked eigenstate (start idx={ref_index}), g={g}')
    plt.legend()
    plt.show()


In [22]:
# Widgets for tracked eigenstates

N = 6
omega_list = np.linspace(0.1, 3.0, 80)
labels = basis_labels(N)

g_slider = widgets.FloatSlider(
    value=0.1, min=0.0, max=1.0, step=0.02, description='g'
)

energy_selector = widgets.IntSlider(
    value=0, min=0, max=2*N-1, step=1, description='Start eig'
)

component_selector = widgets.SelectMultiple(
    options=[(labels[i], i) for i in range(2*N)],
    value=(0, N),
    description='Components',
    rows=8
)

widgets.interactive(
    plot_tracked_components,
    N=widgets.fixed(N),
    g=g_slider,
    ref_index=energy_selector,
    selected_components=component_selector,
    omega_list=widgets.fixed(omega_list)
)


interactive(children=(FloatSlider(value=0.1, description='g', max=1.0, step=0.02), IntSlider(value=0, descript…

### Time-Dependent Dynamics

In [23]:
# Operators

def jc_operators(N):
    a  = tensor(identity(2), destroy(N))
    sm = tensor(0.5*(sigmax() - 1j*sigmay()), identity(N))
    sz = tensor(sigmaz(), identity(N))
    return a, sm, sz


In [24]:
# Time-dependent Hamiltonian
def H_static(N, g):
    a, sm, sz = jc_operators(N)
    return 0.5 * sz + g * (sm * a.dag() + sm.dag() * a)

def H_omega(N):
    a, _, _ = jc_operators(N)
    return a.dag() * a

def omega_t(t, args):
    return args['omega_i'] + args['alpha'] * t

def jc_td_hamiltonian(N, g):
    return [
        H_static(N, g),
        [H_omega(N), omega_t]
    ]


In [25]:
# TDSE solver
def solve_tdse(N, g, ref_index, omega_i, alpha, tlist):
    energies, states = jc_model(N, g, omega_i)
    psi0 = states[ref_index]

    args = {'omega_i': omega_i, 'alpha': alpha}
    H_td = jc_td_hamiltonian(N, g)

    return mesolve(H_td, psi0, tlist, [], [], args=args)


In [26]:
# TD occupation plot

def plot_td_components(
    N, g, ref_index, omega_i, alpha, selected_components
):
    tlist = np.linspace(0, 100, 400)
    result = solve_tdse(N, g, ref_index, omega_i, alpha, tlist)

    labels = basis_labels(N)

    plt.figure(figsize=(7,5))
    for i in selected_components:
        y = [
            abs(state.full().flatten()[i])**2
            for state in result.states
        ]
        plt.plot(tlist, y, label=labels[i])

    plt.xlabel('Time')
    plt.ylabel(r'$|c_i(t)|^2$')
    plt.title('Time-dependent JC dynamics')
    plt.legend()
    plt.show()


In [28]:
# TDSE widgets
omega_i_slider = widgets.FloatSlider(
    value=0.5, min=0.1, max=3.0, step=0.05, description=r'ω_i'
)

alpha_slider = widgets.FloatSlider(
    value=0.02, min=0.0, max=0.1, step=0.005, description=r'α'
)

widgets.interactive(
    plot_td_components,
    N=widgets.fixed(N),
    g=g_slider,
    ref_index=energy_selector,
    omega_i=omega_i_slider,
    alpha=alpha_slider,
    selected_components=component_selector
)


interactive(children=(FloatSlider(value=0.32, description='g', max=1.0, step=0.02), IntSlider(value=0, descrip…